# EPUB → Audiobook

Convert `.epub` books to `.m4b` audiobooks using:

1. **ebooklib** — parse EPUB structure and metadata
2. **Qwen2.5-7B-Instruct** (Hugging Face, 4-bit) — two-step section classification (opening text → catalog) + optional text cleanup for TTS
3. **Kokoro-82M** (Hugging Face) — neural text-to-speech (CPU by default when GPU cuFFT is broken)

**Pipeline**

```
EPUB → catalog → LLM opening analysis → LLM chapter pick → chunks → LLM cleanup → Kokoro TTS → chapter MP3 → M4B
```

**Assumptions (change in the config cell):**

- Input: `experiments/epub-to-audiobook/data/**/*.epub` (or set `EPUB_PATH`)
- Output: `experiments/epub-to-audiobook/data/audio/<book_title>/`
- English books, British English Kokoro voice (`bf_emma`)
- ~90 words per chunk (Kokoro hard limit: **510 phoneme characters** per inference)

**VRAM note (RTX 3080 10 GB):** the LLM is unloaded before Kokoro loads to avoid OOM.

**TTS device:** Kokoro uses `torch.stft` internally. If GPU cuFFT fails in your environment, Step 10 auto-falls back to CPU (`KOKORO_DEVICE = "auto"`).


## Step 0 — Install dependencies

From repo root with devenv: `uv sync --group epub-audiobook` (CUDA torch via pyproject).

System tools (`espeak-ng`, `ffmpeg`) are provided by `devenv.nix`.

In [1]:
import sys
from pathlib import Path

from shared.cuda_bootstrap import ensure_cuda_driver, ensure_torch_compat, ensure_triton_compat

ensure_cuda_driver()
ensure_torch_compat()
ensure_triton_compat()

NOTEBOOK_DIR = Path.cwd()
print(f"Python: {sys.executable}")

# Prefer uv sync from repo root; fall back to pip for notebook-only use.
try:
    import ebooklib  # noqa: F401
except ImportError:
    %pip install -q torch --index-url https://download.pytorch.org/whl/cu124
    %pip install -q -r "{NOTEBOOK_DIR / 'requirements.txt'}"

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

Python: /home/michiel/Code/ai-home-lab/.devenv/state/venv/bin/python


True

## Step 1 — Configuration

In [2]:
from pathlib import Path

# --- paths (experiment-local data folder) ---
DATA_DIR = NOTEBOOK_DIR / "data"
EPUB_DIR = DATA_DIR   / "epub"        # searches here + subfolders for .epub
AUDIO_ROOT = DATA_DIR / "audio"

# --- text chunking ---
# Kokoro hard limit: 510 tokens
WORDS_PER_CHUNK = 250

# --- Hugging Face models (local CUDA) ---
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# British English (required for bf_* voices)
KOKORO_LANG = "b"                     
# Options are: bf_alice, bf_emma, bf_isabella, bf_lily, bm_daniel, bm_fable, bm_george, bm_lewis
KOKORO_VOICE = "bf_emma"
# 1.0 = default; lower = slower narration
KOKORO_SPEED = 0.92                  

LLM_CLEAN_PROMPT = (
    "Clean this text for text-to-speech. Remove URLs, expand abbreviations, "
    "and improve phonetic readability. Return only the cleaned text, no commentary.\n\n"
    "Text:\n{text}"
)

# --- processing toggles ---
RUN_LLM_CLEANUP = True                # set False to skip LLM and use raw chunks

# Chapter selection — pure LLM (Step 6 opening analysis → Step 7 final catalog)
KEEP_CHAPTER_INDICES = None           # manual override, e.g. [4, 5, 6]; None = use LLM
CHAPTER_OPENING_WORDS = 250           # words from each section sent to LLM in Step 6
OPENING_ANALYSIS_BATCH_SIZE = 4       # sections per LLM call in Step 6
LLM_INCLUDE_INTRO = True              # keep introduction/preface for narration
LLM_INCLUDE_APPENDIX = False          # skip appendix / supplement back matter
SHOW_CHAPTER_CATALOG = True

MAX_CHAPTERS = None                      # limit chapters *after* skips; None = all remaining
MAX_CHUNKS_PER_CHAPTER = None            # set e.g. 2 to test TTS on first chunks only; None = all chunks

DEVICE = "cuda"                      # LLM inference device
KOKORO_DEVICE = "auto"               # "auto" | "cuda" | "cpu" — auto picks CPU when GPU cuFFT fails

## Step 2 — Load EPUB file

In [3]:
import ebooklib
from ebooklib import epub

def resolve_epub_path(epub_dir: Path) -> Path:
    if not epub_dir.exists():
        raise FileNotFoundError(
            f"No EPUB directory at {epub_dir.resolve()}. "
            "Create it and add a .epub file."
        )

    epubs = sorted(epub_dir.rglob("*.epub"))
    if not epubs:
        raise FileNotFoundError(f"No .epub files found under {epub_dir.resolve()}")
    if len(epubs) > 1:
        print(f"Multiple EPUBs found, using: {epubs[0].name}")
    return epubs[0]


epub_file = resolve_epub_path(EPUB_DIR)
book = epub.read_epub(str(epub_file))
print(f"Loaded: {epub_file}")

Multiple EPUBs found, using: Against Democracy - Jason Brennan.epub
Loaded: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/epub/Against Democracy - Jason Brennan.epub


## Step 3 — Extract book metadata

In [4]:
import re
from dataclasses import dataclass

from bs4 import BeautifulSoup


def slugify(text: str, max_len: int = 80) -> str:
    text = text.strip().lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text).strip("_")
    return text[:max_len] or "untitled"


@dataclass
class BookMetadata:
    title: str
    author: str
    language: str
    cover_bytes: bytes | None
    output_dir: Path


def get_metadata(epub_book) -> BookMetadata:
    title = epub_book.get_metadata("DC", "title")
    author = epub_book.get_metadata("DC", "creator")
    language = epub_book.get_metadata("DC", "language")

    title = title[0][0] if title else "Unknown Title"
    author = author[0][0] if author else "Unknown Author"
    language = language[0][0] if language else "en"

    cover_bytes = None
    for item in epub_book.get_items_of_type(ebooklib.ITEM_COVER):
        cover_bytes = item.get_content()
        break
    if cover_bytes is None:
        for item in epub_book.get_items_of_type(ebooklib.ITEM_IMAGE):
            if "cover" in item.get_name().lower():
                cover_bytes = item.get_content()
                break

    output_dir = AUDIO_ROOT / slugify(title)
    output_dir.mkdir(parents=True, exist_ok=True)

    return BookMetadata(
        title=title,
        author=author,
        language=language,
        cover_bytes=cover_bytes,
        output_dir=output_dir,
    )


metadata = get_metadata(book)
print(f"Title:   {metadata.title}")
print(f"Author:  {metadata.author}")
print(f"Lang:    {metadata.language}")
print(f"Cover:   {'yes' if metadata.cover_bytes else 'no'}")
print(f"Output:  {metadata.output_dir}")

Title:   Against Democracy
Author:  Jason Brennan
Lang:    en
Cover:   yes
Output:  /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy


## Step 4 — Extract EPUB sections (catalog)

Lists every readable section. **Steps 6–7** pick which to narrate (LLM, using opening text).

In [5]:
from dataclasses import dataclass


@dataclass
class Chapter:
    index: int
    title: str
    text: str
    slug: str


def html_to_text(content: bytes | str) -> str:
    # EPUB chapter files are XHTML (XML), not plain HTML.
    try:
        soup = BeautifulSoup(content, features="xml")
    except Exception:
        soup = BeautifulSoup(content, "lxml")

    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


def extract_all_chapters(epub_book) -> list[Chapter]:
    chapters: list[Chapter] = []
    chapter_idx = 0

    for item in epub_book.get_items():
        if item.get_type() != ebooklib.ITEM_DOCUMENT:
            continue

        text = html_to_text(item.get_content())
        if len(text.split()) < 30:
            continue

        name = Path(item.get_name()).stem.replace("_", " ").strip()
        title = name.title() if name else f"Chapter {chapter_idx + 1}"

        chapters.append(
            Chapter(
                index=chapter_idx,
                title=title,
                text=text,
                slug=slugify(title),
            )
        )
        chapter_idx += 1

    if not chapters:
        raise ValueError("No readable chapters found in EPUB")

    return chapters


all_chapters = extract_all_chapters(book)
chapters = None
chapter_opening_analyses = None
chapter_decisions = None

if SHOW_CHAPTER_CATALOG:
    print(f"EPUB catalog ({len(all_chapters)} sections):")
    for ch in all_chapters:
        words = len(ch.text.split())
        print(f"  [{ch.index:2d}] {ch.title} ({words} words)")

if KEEP_CHAPTER_INDICES:
    chapters = [ch for ch in all_chapters if ch.index in KEEP_CHAPTER_INDICES]
    if MAX_CHAPTERS is not None:
        chapters = chapters[:MAX_CHAPTERS]
    print(f"\nManual selection: {len(chapters)} section(s)")
else:
    print("\nRun Steps 5 → 6 → 7 to classify sections (opening analysis → final catalog).")

EPUB catalog (15 sections) — filter mode: hybrid
  [ 0] Part0002 (147 words)
  [ 1] Part0003 (65 words)
  [ 2] Part0004 (922 words)
  [ 3] Part0006 (7582 words)
  [ 4] Part0007 (11104 words)
  [ 5] Part0008 (6784 words)
  [ 6] Part0009 (13880 words)
  [ 7] Part0010 (10212 words)
  [ 8] Part0011 (11018 words)
  [ 9] Part0012 (11117 words)
  [10] Part0013 (9614 words)
  [11] Part0014 (5154 words)
  [12] Part0015 (6134 words)
  [13] Part0016 (4717 words)
  [14] Part0017 (3547 words)

Run Step 5 (load LLM) then Step 6 to classify sections.


## Step 5 — Load Qwen LLM (Hugging Face, 4-bit on CUDA)

Loaded once for **chapter classification** (Steps 6–7) and optional **text cleanup** (Step 9).

In [6]:
import gc
import sys

from shared.cuda_bootstrap import ensure_cuda_driver, ensure_torch_compat, ensure_triton_compat

ensure_cuda_driver()
ensure_torch_compat()
ensure_triton_compat()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

llm_model = None
llm_tokenizer = None

need_llm = RUN_LLM_CLEANUP or not KEEP_CHAPTER_INDICES

if need_llm:
    if not torch.cuda.is_available():
        raise RuntimeError(
            f"CUDA not available — use kernel 'ai-home-lab (devenv)' and restart. Python: {sys.executable}"
        )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.float16,
    )
    llm_model.eval()
    print(f"Loaded {LLM_MODEL_ID} on {DEVICE}")
else:
    print("Skipping LLM load (keywords/manual mode, RUN_LLM_CLEANUP=False)")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-7B-Instruct on cuda


## Step 6 — Analyze section openings (LLM)

For each EPUB section, Qwen reads the **first ~250 words** (not just the file title) and
classifies whether it is narratively important or front/back matter (TOC, copyright, etc.).

In [7]:
import importlib

import shared.chapter_catalog as chapter_catalog

importlib.reload(chapter_catalog)
from shared.chapter_catalog import analyze_chapter_openings_with_llm

if chapters is not None:
    print("Skipping Step 6 — KEEP_CHAPTER_INDICES set in Step 4.")
elif llm_model is None or llm_tokenizer is None:
    raise RuntimeError("Run Step 5 first — LLM not loaded.")
else:
    chapter_opening_analyses = analyze_chapter_openings_with_llm(
        llm_model,
        llm_tokenizer,
        all_chapters,
        book_title=metadata.title,
        opening_words=CHAPTER_OPENING_WORDS,
        include_intro=LLM_INCLUDE_INTRO,
        include_appendix=LLM_INCLUDE_APPENDIX,
        batch_size=OPENING_ANALYSIS_BATCH_SIZE,
    )

    print(f"Opening analysis ({len(chapter_opening_analyses)} sections):")
    for a in chapter_opening_analyses:
        ch = next(c for c in all_chapters if c.index == a.index)
        flag = "KEEP" if a.narratively_important else "SKIP"
        print(f"  [{a.index:2d}] {flag:4s} {ch.title}")
        print(f"         {a.section_type} — {a.reason}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Keyword pre-filter (hints for LLM):
  [ 0] SKIP Part0002 — index 0
  [ 1] KEEP Part0003
  [ 2] KEEP Part0004
  [ 3] KEEP Part0006
  [ 4] KEEP Part0007
  [ 5] KEEP Part0008
  [ 6] KEEP Part0009
  [ 7] KEEP Part0010
  [ 8] KEEP Part0011
  [ 9] KEEP Part0012
  [10] KEEP Part0013
  [11] KEEP Part0014
  [12] KEEP Part0015
  [13] KEEP Part0016
  [14] KEEP Part0017

Hybrid chapter plan (15 sections):
  [ 0] SKIP Part0002 — copyright [kw:skip]
  [ 1] KEEP Part0003 — chapter [kw:keep]
  [ 2] KEEP Part0004 — preface [kw:keep]
  [ 3] KEEP Part0006 — chapter [kw:keep]
  [ 4] KEEP Part0007 — chapter [kw:keep]
  [ 5] KEEP Part0008 — chapter [kw:keep]
  [ 6] KEEP Part0009 — chapter [kw:keep]
  [ 7] KEEP Part0010 — chapter [kw:keep]
  [ 8] KEEP Part0011 — chapter [kw:keep]
  [ 9] KEEP Part0012 — chapter [kw:keep]
  [10] KEEP Part0013 — chapter [kw:keep]
  [11] KEEP Part0014 — chapter [kw:keep]
  [12] SKIP Part0015 — notes (LLM skip; keyword had keep) [kw:keep]
  [13] SKIP Part0016 — bibliography (LLM 

## Step 7 — Final chapter selection (LLM)

Uses the Step 6 opening analyses plus catalog metadata to make the final keep/skip list.
Override with `KEEP_CHAPTER_INDICES` in config if needed.

In [ ]:
import importlib

import shared.chapter_catalog as chapter_catalog

importlib.reload(chapter_catalog)
from shared.chapter_catalog import (
    apply_chapter_decisions,
    build_catalog_entries,
    classify_chapters_with_llm,
)

if chapters is not None:
    print("Using manual KEEP_CHAPTER_INDICES from Step 4.")
elif chapter_opening_analyses is None:
    raise RuntimeError("Run Step 6 first — opening analysis missing.")
elif llm_model is None or llm_tokenizer is None:
    raise RuntimeError("Run Step 5 first — LLM not loaded.")
else:
    entries = build_catalog_entries(all_chapters)
    decisions = classify_chapters_with_llm(
        llm_model,
        llm_tokenizer,
        book_title=metadata.title,
        entries=entries,
        opening_analyses=chapter_opening_analyses,
        include_intro=LLM_INCLUDE_INTRO,
        include_appendix=LLM_INCLUDE_APPENDIX,
    )
    chapters, chapter_decisions = apply_chapter_decisions(
        all_chapters,
        decisions,
        keep_indices_override=KEEP_CHAPTER_INDICES,
        max_chapters=MAX_CHAPTERS,
    )

    print(f"Final chapter plan ({len(all_chapters)} sections):")
    for ch in all_chapters:
        d = chapter_decisions.get(ch.index)
        o = next((a for a in chapter_opening_analyses if a.index == ch.index), None)
        marker = "KEEP" if d and d.keep else "SKIP"
        reason = d.reason if d else "no decision"
        opening_note = f" [opening: {o.section_type}]" if o else ""
        print(f"  [{ch.index:2d}] {marker:4s} {ch.title} — {reason}{opening_note}")

    keep_indices = [ch.index for ch in chapters]
    print(f"\nkeep_indices = {keep_indices}")

print(f"\nProcessing {len(chapters)} chapter(s):")
for ch in chapters[:15]:
    print(f"  [{ch.index}] {ch.title} ({len(ch.text.split())} words)")
if len(chapters) > 15:
    print("  ...")

## Step 8 — Chunk chapters into word-sized pieces

Kokoro splits on **510 phoneme characters** internally; we pre-chunk at ~90 words so each chunk maps to one WAV file and stays safely under the limit.

In [8]:
from dataclasses import dataclass


@dataclass
class TextChunk:
    chapter_index: int
    chapter_title: str
    chapter_slug: str
    chunk_index: int
    text: str


def chunk_words(text: str, words_per_chunk: int) -> list[str]:
    words = text.split()
    if not words:
        return []

    chunks = []
    for start in range(0, len(words), words_per_chunk):
        chunk = " ".join(words[start : start + words_per_chunk]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks


def build_chunks(chapter_list: list[Chapter], words_per_chunk: int) -> list[TextChunk]:
    all_chunks: list[TextChunk] = []
    for chapter in chapter_list:
        raw_chunks = chunk_words(chapter.text, words_per_chunk)
        if MAX_CHUNKS_PER_CHAPTER is not None:
            raw_chunks = raw_chunks[:MAX_CHUNKS_PER_CHAPTER]

        for i, chunk_text in enumerate(raw_chunks):
            all_chunks.append(
                TextChunk(
                    chapter_index=chapter.index,
                    chapter_title=chapter.title,
                    chapter_slug=chapter.slug,
                    chunk_index=i,
                    text=chunk_text,
                )
            )
    return all_chunks


if chapters is None:
    raise RuntimeError("Run Step 7 first — chapters not selected yet.")

chunks = build_chunks(chapters, WORDS_PER_CHUNK)
print(f"Created {len(chunks)} chunk(s) @ {WORDS_PER_CHUNK} words each")
print("Sample:", chunks[0].text[:200], "...")

Created 356 chunk(s) @ 250 words each
Sample: CONTENTS Preface and Acknowledgments vii 1 Hobbits and Hooligans 1 2 Ignorant, Irrational, Misinformed Nationalists 23 3 Political Participation Corrupts 54 4 Politics Doesn’t Empower You or Me 74 5 P ...


## _(LLM load moved to Step 5 — skip this cell)_

In [9]:
# LLM is loaded in Step 5.
if (RUN_LLM_CLEANUP or not KEEP_CHAPTER_INDICES) and llm_model is None:
    raise RuntimeError("Run Step 5 to load the LLM.")
print("LLM ready." if llm_model is not None else "No LLM loaded (manual chapter list, no cleanup).")

LLM ready.


## Step 9 — Clean text chunks with the LLM

In [10]:
import gc

import torch
from dataclasses import dataclass

if RUN_LLM_CLEANUP and ("llm_model" not in globals() or "llm_tokenizer" not in globals()):
    raise RuntimeError("Run Step 5 first — llm_model and llm_tokenizer are not loaded.")
if RUN_LLM_CLEANUP and (llm_model is None or llm_tokenizer is None):
    raise RuntimeError("Run Step 5 first — LLM load did not complete.")


@dataclass
class CleanedChunk:
    chapter_index: int
    chapter_title: str
    chapter_slug: str
    chunk_index: int
    raw_text: str
    cleaned_text: str


def clean_text_with_llm(text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You prepare book text for text-to-speech narration.",
        },
        {"role": "user", "content": LLM_CLEAN_PROMPT.format(text=text)},
    ]
    prompt = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            temperature=0.0,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()


cleaned_chunks: list[CleanedChunk] = []

for i, chunk in enumerate(chunks):
    if RUN_LLM_CLEANUP:
        cleaned = clean_text_with_llm(chunk.text)
    else:
        cleaned = chunk.text

    cleaned_chunks.append(
        CleanedChunk(
            chapter_index=chunk.chapter_index,
            chapter_title=chunk.chapter_title,
            chapter_slug=chunk.chapter_slug,
            chunk_index=chunk.chunk_index,
            raw_text=chunk.text,
            cleaned_text=cleaned,
        )
    )
    print(f"[{i + 1}/{len(chunks)}] cleaned chunk {chunk.chunk_index} of '{chunk.chapter_title}'")

# Free VRAM before loading Kokoro
if llm_model is not None:
    del llm_model
    del llm_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Unloaded LLM to free GPU memory for Kokoro")

[1/356] cleaned chunk 0 of 'Part0003'
[2/356] cleaned chunk 0 of 'Part0004'
[3/356] cleaned chunk 1 of 'Part0004'
[4/356] cleaned chunk 2 of 'Part0004'
[5/356] cleaned chunk 3 of 'Part0004'
[6/356] cleaned chunk 0 of 'Part0006'
[7/356] cleaned chunk 1 of 'Part0006'
[8/356] cleaned chunk 2 of 'Part0006'
[9/356] cleaned chunk 3 of 'Part0006'
[10/356] cleaned chunk 4 of 'Part0006'
[11/356] cleaned chunk 5 of 'Part0006'
[12/356] cleaned chunk 6 of 'Part0006'
[13/356] cleaned chunk 7 of 'Part0006'
[14/356] cleaned chunk 8 of 'Part0006'
[15/356] cleaned chunk 9 of 'Part0006'
[16/356] cleaned chunk 10 of 'Part0006'
[17/356] cleaned chunk 11 of 'Part0006'
[18/356] cleaned chunk 12 of 'Part0006'
[19/356] cleaned chunk 13 of 'Part0006'
[20/356] cleaned chunk 14 of 'Part0006'
[21/356] cleaned chunk 15 of 'Part0006'
[22/356] cleaned chunk 16 of 'Part0006'
[23/356] cleaned chunk 17 of 'Part0006'
[24/356] cleaned chunk 18 of 'Part0006'
[25/356] cleaned chunk 19 of 'Part0006'
[26/356] cleaned chunk 2

## 7.2 Future — Voice direction tags (optional)

Pass cleaned chunks through the LLM again to inject pause/emphasis/speed hints before TTS. Not implemented yet.


## Step 10 — Load Kokoro TTS (Hugging Face)

Uses [`hexgrad/Kokoro-82M`](https://huggingface.co/hexgrad/Kokoro-82M) via the `kokoro` package.

**Other TTS options for this stack** (long-form English narration):

| Model | Fit | Notes |
|-------|-----|-------|
| **Kokoro-82M** (current) | Best lightweight HF option | ~82M params, British voices, runs on CPU when GPU cuFFT is broken |
| [**Piper**](https://huggingface.co/rhasspy/piper-voices) | Best CPU reliability | ONNX (no PyTorch FFT), very fast; slightly less natural than larger models |
| [**Parler-TTS mini**](https://huggingface.co/parler-tts/parler-tts-mini-v1) | Native `transformers` | Describe voice in text; ~880M — heavier, may hit same cuFFT issue on GPU |
| [**MeloTTS**](https://huggingface.co/myshell-ai/MeloTTS-English) | Good HF alternative | Multi-speaker English; moderate size |
| **XTTS-v2** | Highest quality | ~2GB+, slow; Coqui licensing/maintenance caveats |

For prototyping on an RTX 3080 with a Qwen LLM pass first, **Kokoro on CPU** or **Piper** are the most practical. Swap only if you need higher fidelity and can accept slower/heavier inference.

In [11]:
import gc
import importlib

import numpy as np
import soundfile as sf
from kokoro import KPipeline

import shared.cuda_bootstrap as cuda_bootstrap

importlib.reload(cuda_bootstrap)
cuda_fft_available = cuda_bootstrap.cuda_fft_available

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if KOKORO_DEVICE == "auto":
    kokoro_device = "cuda" if cuda_fft_available() else "cpu"
else:
    kokoro_device = KOKORO_DEVICE

kokoro_pipeline = KPipeline(lang_code=KOKORO_LANG, device=kokoro_device)
SAMPLE_RATE = 24_000
print(
    f"Loaded Kokoro on {kokoro_device} "
    f"(lang={KOKORO_LANG}, voice={KOKORO_VOICE}, speed={KOKORO_SPEED})"
)
if kokoro_device == "cpu":
    print("GPU cuFFT unavailable — Kokoro runs on CPU (~82M params).")

/home/michiel/Code/ai-home-lab/.devenv/state/venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/michiel/Code/ai-home-lab/.devenv/state/venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded Kokoro on cpu (lang=b, voice=bf_emma, speed=0.92)
GPU cuFFT unavailable — Kokoro runs on CPU (~82M params).


## Step 11 — Generate WAV files per chunk

In [ ]:
from dataclasses import dataclass


@dataclass
class AudioChunk:
    chapter_index: int
    chapter_title: str
    chapter_slug: str
    chunk_index: int
    wav_path: Path


def synthesize_to_wav(text: str, wav_path: Path) -> None:
    wav_path.parent.mkdir(parents=True, exist_ok=True)

    audio_parts = []
    generator = kokoro_pipeline(text, voice=KOKORO_VOICE, speed=KOKORO_SPEED)
    for _, _, audio in generator:
        if audio is None:
            continue
        if hasattr(audio, "detach"):
            audio = audio.detach().cpu().numpy()
        audio_parts.append(np.asarray(audio, dtype=np.float32))

    if not audio_parts:
        raise RuntimeError(f"Kokoro produced no audio for: {wav_path}")

    merged = np.concatenate(audio_parts)
    sf.write(wav_path, merged, SAMPLE_RATE)


audio_chunks: list[AudioChunk] = []

for i, chunk in enumerate(cleaned_chunks):
    chapter_dir = metadata.output_dir / "wav"/ chunk.chapter_slug
    wav_path = chapter_dir / f"{chunk.chunk_index:04d}.wav"

    synthesize_to_wav(chunk.cleaned_text, wav_path)
    audio_chunks.append(
        AudioChunk(
            chapter_index=chunk.chapter_index,
            chapter_title=chunk.chapter_title,
            chapter_slug=chunk.chapter_slug,
            chunk_index=chunk.chunk_index,
            wav_path=wav_path,
        )
    )
    print(f"[{i + 1}/{len(cleaned_chunks)}] {wav_path}")


def preview_audio_chunks(
    chunks: list[AudioChunk],
    texts: list,
) -> None:
    """Interactive WAV preview — one dropdown, player, and text snippet per chunk."""
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    if not chunks:
        print("No audio chunks to preview.")
        return

    labels = [
        f"Ch{ac.chapter_index} #{ac.chunk_index:04d} — {ac.chapter_title[:48]}"
        for ac in chunks
    ]
    idx_by_label = {label: i for i, label in enumerate(labels)}

    dropdown = widgets.Dropdown(options=labels, description="Chunk:", layout=widgets.Layout(width="95%"))
    audio_box = widgets.Output()
    text_box = widgets.HTML()

    def show_chunk(_change=None):
        i = idx_by_label[dropdown.value]
        ac = chunks[i]
        snippet = texts[i].cleaned_text if hasattr(texts[i], "cleaned_text") else str(texts[i])
        if len(snippet) > 600:
            snippet = snippet[:600] + "…"

        with audio_box:
            clear_output(wait=True)
            display(
                widgets.Audio(
                    value=ac.wav_path.read_bytes(),
                    format="audio/wav",
                    autoplay=False,
                )
            )
        text_box.value = (
            f"<b>{ac.chapter_title}</b> (chunk {ac.chunk_index})<br>"
            f"<pre style='white-space:pre-wrap;margin-top:0.5em'>{snippet}</pre>"
        )

    dropdown.observe(show_chunk, names="value")
    show_chunk()
    display(widgets.VBox([dropdown, audio_box, text_box]))


preview_audio_chunks(audio_chunks, cleaned_chunks)

[1/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0003/0000.wav
[2/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0004/0000.wav
[3/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0004/0001.wav
[4/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0004/0002.wav
[5/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0004/0003.wav
[6/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0006/0000.wav
[7/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0006/0001.wav
[8/356] /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/against_democracy/wav/part0006/0002.wav
[9/356] /home/michiel/Code/ai-home-lab/e

## Step 12 — Stitch chunk WAVs into chapter MP3 files

In [ ]:
import importlib
from collections import defaultdict

import shared.ffmpeg_utils as ffmpeg_utils

importlib.reload(ffmpeg_utils)
from shared.ffmpeg_utils import concat_audio_files

chapter_mp3_paths: dict[str, Path] = {}
by_chapter: dict[str, list[AudioChunk]] = defaultdict(list)

for ac in audio_chunks:
    by_chapter[ac.chapter_slug].append(ac)

for chapter_slug, ac_list in by_chapter.items():
    ac_list.sort(key=lambda x: x.chunk_index)
    wav_paths = [ac.wav_path for ac in ac_list]

    mp3_path = metadata.output_dir / "mp3" / f"{chapter_slug}.mp3"
    concat_audio_files(
        wav_paths,
        mp3_path,
        audio_codec="libmp3lame",
        bitrate="128k",
    )
    chapter_mp3_paths[chapter_slug] = mp3_path
    print(f"Chapter MP3: {mp3_path} ({len(wav_paths)} chunk(s))")

Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_itr_r1.mp3 (18 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_c001_r1.mp3 (61 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_c002_r1.mp3 (57 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_c003_r1.mp3 (61 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_c004_r1.mp3 (48 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_9780399178757_epub3_c005_r1.mp3 (46 chunk(s))
Chapter MP3: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/mp3/hall_97803991787

## Step 13 — Stitch chapters into a single M4B audiobook

Concatenates chapter MP3s, embeds **chapter markers**, and writes EPUB metadata (title, author, language, cover art) into the final file.

In [ ]:
import importlib

import shared.audiobook_metadata as audiobook_metadata
import shared.ffmpeg_utils as ffmpeg_utils

importlib.reload(ffmpeg_utils)
importlib.reload(audiobook_metadata)
from shared.audiobook_metadata import finish_m4b_audiobook
from shared.ffmpeg_utils import concat_audio_files

ordered_chapters = [ch for ch in chapters if ch.slug in chapter_mp3_paths]
chapter_files = [chapter_mp3_paths[ch.slug] for ch in ordered_chapters]

if not chapter_files:
    raise RuntimeError("No chapter MP3 files to combine")

m4b_path = metadata.output_dir / f"{slugify(metadata.title)}.m4b"

concat_audio_files(
    chapter_files,
    m4b_path,
    audio_codec="aac",
    bitrate="64k",
    extra_args=["-movflags", "+faststart"],
)

chapter_audio = [
    (ch.title, chapter_mp3_paths[ch.slug]) for ch in ordered_chapters
]
finish_m4b_audiobook(
    m4b_path,
    title=metadata.title,
    author=metadata.author,
    language=metadata.language,
    cover_bytes=metadata.cover_bytes,
    chapter_audio=chapter_audio,
)

print(f"Audiobook ready: {m4b_path}")
print(f"  title:    {metadata.title}")
print(f"  author:   {metadata.author}")
print(f"  language: {metadata.language}")
print(f"  cover:    {'yes' if metadata.cover_bytes else 'no'}")
print(f"  chapters: {len(chapter_files)} (with markers)")

Audiobook ready: /home/michiel/Code/ai-home-lab/experiments/epub-to-audiobook/data/audio/adhd_20/adhd_20.m4b
  title:    ADHD 2.0
  author:   Edward M. Hallowell, M.D.
  language: en-US
  cover:    yes
  chapters: 10 (with markers)
